In [ ]:
from pathlib import Path
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from skimage.morphology import erosion, dilation
from skimage.segmentation import watershed
from skimage.io import imread, imsave
from scipy.ndimage import gaussian_gradient_magnitude
from tifffile import imwrite

from snap_to_edge import snap_labels_to_edge
from calmutils.morphology.structuring_elements import hypersphere_centered
from io_helpers import imsave_nowarnings
from visualization_utils import get_segmentation_visualization


def process_single_image(mask_path, raw_path, out_path, vis_out_path, edge_filter_sigmas, radius_morphology):
    
    # load
    labels = imread(mask_path)
    img = imread(raw_path)

    # filter to enhance edges
    edge_img = gaussian_gradient_magnitude(img.astype(float), edge_filter_sigmas)

    # refine labels
    labels_snap = snap_labels_to_edge(labels, edge_img, radius_morphology)
    
    # save as int (so it will be recognized as label by napari), compress to save significant space
    imwrite(out_path, labels_snap.astype(int), compression=5)

    # save visualization if necessary
    if vis_out_path is not None:
        rgb_vis = get_segmentation_visualization(labels_snap, img)
        imsave_nowarnings(vis_out_path, rgb_vis)
    

In [ ]:
in_path = '/Volumes/nn/Julia Vogtmann/Microscopy/25JV_006'

mask_subdirectory = 'segmentation_nucleoli'
raw_subdirectory = 'tif'
mask_suffix = '_segmented'

out_subdirectory = 'segmentation_nucleoli_edgesnap'
visualization_subdirectory = 'vis'

edge_filter_sigmas = (2, 1, 1)
radius_morphology = 2

In [ ]:
# get all TIFF files in mask directory
mask_files = sorted((Path(in_path) / mask_subdirectory).glob('[!.]*.tif'))

# get raw files by dropping suffix from mask filenames
raw_files = [Path(in_path) / raw_subdirectory / mask_file.name.replace(mask_suffix, '') for mask_file in mask_files]

raw_files

In [ ]:
out_path = (Path(in_path) / out_subdirectory)
if not out_path.exists():
    out_path.mkdir(parents=True)

if (visualization_subdirectory is not None) and not (out_path / visualization_subdirectory).exists():
    (out_path / visualization_subdirectory).mkdir()

with ThreadPoolExecutor() as tpe:
    futures = []

    # enqueue
    for mask_file, raw_file in zip(mask_files, raw_files):
        out_file = out_path / mask_file.name

        vis_file = None
        if visualization_subdirectory is not None:
            vis_file = (out_path / visualization_subdirectory / mask_file.name.replace('.tif', '.png'))
        
        f = tpe.submit(process_single_image, mask_file, raw_file, out_file, vis_file, edge_filter_sigmas, radius_morphology)
        futures.append(f)
        
    # notify upon completion
    for f, mask_file in zip(futures, mask_files):
        f.result()
        print(f'processed {str(mask_file)}.')